In [0]:
%sql
use catalog ecom_dev;
-- 1. Create a schema for pipeline control plane / configuration
CREATE SCHEMA IF NOT EXISTS ecom_dev.pipeline_config;

-- 2. Create the metadata configuration table
CREATE TABLE IF NOT EXISTS ecom_dev.pipeline_config.ingestion_metadata (
    id BIGINT GENERATED ALWAYS AS IDENTITY,
    target_table STRING NOT NULL COMMENT 'Name of the unified streaming table to target',
    flow_name STRING NOT NULL COMMENT 'Unique identifier for the append flow',
    source_path STRING NOT NULL COMMENT 'Volume or object storage directory where raw files land',
    source_format STRING NOT NULL COMMENT 'File format (json, csv, parquet, etc.)',
    channel STRING NOT NULL COMMENT 'Logical source tag injected into records (e.g., WEB, MOBILE)',
    cluster_keys STRING COMMENT 'Comma-separated columns for Liquid Clustering',
    schema_ddl STRING NOT NULL COMMENT 'Explicit SQL DDL column schema contract',
    is_active BOOLEAN NOT NULL DEFAULT true COMMENT 'Flag to pause or activate the flow dynamically',
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'Metadata registry driving dynamic declarative ingestion pipelines';

In [0]:
%sql
INSERT INTO ecom_dev.pipeline_config.ingestion_metadata 
(target_table, flow_name, source_path, source_format, channel, cluster_keys, schema_ddl, is_active)
VALUES
(
    'ecom_dev.raw.bronze_orders',
    'ingest_web_orders',
    '/Volumes/ecom_dev/raw/landing/orders_web',
    'json',
    'WEB',
    'country,customer_id',
    'order_id STRING, customer_id STRING, email STRING, phone STRING, order_ts TIMESTAMP, amount DOUBLE, currency STRING, country STRING',
    true
),
(
    'ecom_dev.raw.bronze_orders',
    'ingest_mobile_orders',
    '/Volumes/ecom_dev/raw/landing/orders_mobile',
    'json',
    'MOBILE',
    'country,customer_id',
    'order_id STRING, customer_id STRING, email STRING, phone STRING, order_ts TIMESTAMP, amount DOUBLE, currency STRING, country STRING',
    true
);

-- Verify the metadata
SELECT * FROM ecom_dev.pipeline_config.ingestion_metadata WHERE is_active = true;



In [0]:
# %sql
# DROP TABLE IF EXISTS workspace.default.event_log_8613a716_7408_4656_91ff_13efed934425;

In [0]:
%sql
SELECT * FROM ecom_dev.pipeline_config.ingestion_metadata WHERE is_active = true;

### cleanup task if table already present

In [0]:
# %sql
# -- Syntax: DROP TABLE IF EXISTS <catalog>.<schema>.<table_name>;

# -- Example: Dropping a table from the 'raw' schema
# DROP TABLE IF EXISTS ecom_dev.raw.bronze_orders;
# DROP TABLE IF EXISTS ecom_dev.raw.bronze_customers_cdc;
# DROP TABLE IF EXISTS ecom_dev.raw.bronze_multiplex_events;
# DROP TABLE IF EXISTS ecom_dev.raw.bronze_payments;

# -- Example: Dropping a table from the 'audit' schema
# DROP TABLE IF EXISTS ecom_dev.audit.reprocess_queue;
# DROP TABLE IF EXISTS ecom_dev.audit.file_manifest;
# DROP TABLE IF EXISTS ecom_dev.audit.load_source_config;
# DROP TABLE IF EXISTS ecom_dev.audit.reconcile_exception;
# DROP TABLE IF EXISTS ecom_dev.audit.reconcile_run;
# DROP TABLE IF EXISTS ecom_dev.audit.schema_registry;